In [1]:
## Combine Data and Load Libraries

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
import os
import glob

# Load all CSV files from the 'simulation_results' folder
folder_path = os.path.join(os.getcwd(), 'simulation_results')
file_paths = glob.glob(os.path.join(folder_path, '*.csv'))

df_list = []
for file in file_paths:
    df = pd.read_csv(file)
    # Normalize column names to lowercase for consistency
    df.columns = [c.lower() for c in df.columns]
    # Keep only columns starting with "feature_" and "algo_safety"
    cols_to_keep = [c for c in df.columns if c.startswith('feature_') or c == 'algo_safety']
    df = df[cols_to_keep]
    # Remove "feature_" prefix from column names
    df.columns = [c.replace('feature_', '') for c in df.columns]
    df_list.append(df)

df = pd.concat(df_list, ignore_index=True)

# Display basic info
print(df.shape)
print(df.head())

(17766, 11)
  direction  cloudiness  precipitation  precipitationdeposits  windintensity  \
0      left          40              0                     60             40   
1     right          40              0                     20             20   
2      left           0             60                      0             80   
3     right           0              0                      0            100   
4     right          40             60                      0             40   

   timeofday  fogdensity  fogdistance  wetness  roadfriction  algo_safety  
0        -90           0           60       80           0.8            0  
1        -30          80           40      100           0.2            0  
2          0           0            0       40           1.0            0  
3        -60          20            0       40           0.6            0  
4        -30          80           60       20           0.8            0  


In [5]:
df.head()

,direction,cloudiness,precipitation,precipitationdeposits,windintensity,timeofday,fogdensity,fogdistance,wetness,roadfriction,algo_safety
0,left,40,0,60,40,-90,0,60,80,0.8,0
1,right,40,0,20,20,-30,80,40,100,0.2,0
2,left,0,60,0,80,0,0,0,40,1.0,0
3,right,0,0,0,100,-60,20,0,40,0.6,0
4,right,40,60,0,40,-30,80,60,20,0.8,0


In [6]:
## Data Preprocessing

In [8]:
# Check for missing values
print(df.isnull().sum())

# Separate features and targets
features = df.drop(['algo_safety'], axis=1)
target_collision = df['algo_safety']

direction                0
cloudiness               0
precipitation            0
precipitationdeposits    0
windintensity            0
timeofday                0
fogdensity               0
fogdistance              0
wetness                  0
roadfriction             0
algo_safety              0
dtype: int64


In [ ]:
## Analyze Influence on Collision_Occurred

### Collision Rate per Factor Value
For each factor, calculate the collision rate for each value. This is more directly useful for importance indices.

In [9]:
# For each factor, group by values and calculate the collision rate
collision_rates = {}
for column in features.columns:
    grouped = df.groupby(column)['algo_safety'].mean()
    collision_rates[column] = grouped
    print(f"Collision rates for {column}:")
    print(grouped)
    print()

Collision rates for direction:
direction
left     0.001380
right    0.000772
Name: algo_safety, dtype: float64

Collision rates for cloudiness:
cloudiness
0      0.001900
20     0.001327
40     0.001005
60     0.000677
80     0.000369
100    0.001105
Name: algo_safety, dtype: float64

Collision rates for precipitation:
precipitation
0      0.000000
20     0.000724
40     0.000733
60     0.000807
80     0.001173
100    0.003579
Name: algo_safety, dtype: float64

Collision rates for precipitationdeposits:
precipitationdeposits
0      0.001562
20     0.001913
40     0.000000
60     0.000747
80     0.001124
100    0.000704
Name: algo_safety, dtype: float64

Collision rates for windintensity:
windintensity
0      0.000000
20     0.001588
40     0.000914
60     0.000357
80     0.002730
100    0.000663
Name: algo_safety, dtype: float64

Collision rates for timeofday:
timeofday
-90    0.001932
-60    0.001732
-30    0.000418
 0     0.000426
 30    0.002701
 60    0.000000
 90    0.000000
Name:

In [10]:
## Transform Collision Rates to Importance Indices

In [11]:
def normalize_to_importance_indices(collision_rates):
    """
    Convert collision rates to importance indices where ALL values sum to 1 total
    """
    # First, calculate the total sum across all parameters and values
    total_sum = 0
    for param, rates in collision_rates.items():
        total_sum += rates.sum()
    
    importance_indices = {}
    
    for param, rates in collision_rates.items():
        # Normalize each value by the grand total
        importance_indices[param] = rates / total_sum
    
    return importance_indices

# Calculate importance indices
final_importance_indices = normalize_to_importance_indices(collision_rates)

# Print results in the desired format
print("# Importance indices based on collision rates:")
print("# Format: Parameter_Value,ImportanceIndex")
print()

for param, importance in final_importance_indices.items():
    line = " ".join(f"{param}_{value},{idx:.6f}" for value, idx in importance.items())
    print(line)


# Importance indices based on collision rates:
# Format: Parameter_Value,ImportanceIndex

direction_left,0.022231 direction_right,0.012431
cloudiness_0,0.030612 cloudiness_20,0.021375 cloudiness_40,0.016184 cloudiness_60,0.010902 cloudiness_80,0.005944 cloudiness_100,0.017793
precipitation_0,0.000000 precipitation_20,0.011656 precipitation_40,0.011810 precipitation_60,0.013006 precipitation_80,0.018899 precipitation_100,0.057653
precipitationdeposits_0,0.025158 precipitationdeposits_20,0.030812 precipitationdeposits_40,0.000000 precipitationdeposits_60,0.012026 precipitationdeposits_80,0.018099 precipitationdeposits_100,0.011344
windintensity_0,0.000000 windintensity_20,0.025585 windintensity_40,0.014724 windintensity_60,0.005745 windintensity_80,0.043982 windintensity_100,0.010685
timeofday_-90,0.031127 timeofday_-60,0.027905 timeofday_-30,0.006731 timeofday_0,0.006858 timeofday_30,0.043502 timeofday_60,0.000000 timeofday_90,0.000000
fogdensity_0,0.003518 fogdensity_20,0.012021 fogden

In [12]:
## Saving th parameters in txt file

In [13]:
with open("parameters_scenario3.txt", "w") as f:
    for param, importance in final_importance_indices.items():
        line = " ".join(f"{param}_{value},{idx:.6f}" for value, idx in importance.items())
        f.write(line + "\n")